# controls

> Buttons, button groups, fields, switches, checkboxes, radios, ranges, and fieldsets

In [ ]:
#| default_exp controls

Every form control we need is already a native HTML tag, and the browser already keeps track of its state. We style those tags directly, and we style their states through the pseudo-classes (`:disabled`, `:user-invalid`, …) and ARIA attributes (`aria-busy`, `aria-pressed`, …) that the browser provides, rather than tracking anything ourselves. Variants are bare attributes.

In [ ]:
from fastcore.utils import *
from fasthtml.common import Div, A, Label, Input, Small, Form, Textarea, Select, Option, Legend
from answerui.registry import reg, hypelt
import answerui.tokens  # installs Registry.context
from answerui.demo import example
from answerui.layout import Stack, Cluster

In [ ]:
from fasthtml.common import *
from answerui.demo import demo
from answerui.registry import reg
from fastcore.test import *
from fasthtml.svg import Path

## Button

We design our buttons with deliberate depth affordances to indicate 'clickability'. A raised face invites a press, and a sunken face shows that a button is already pressed. We keep the overall size of a button fixed while a press happens, moving only its label, bevels and shadows, so that a row of buttons stays level however many of them are down.

Some presses last longer than a click. While a request is in flight, the button carries `aria-busy`, and while a toggle is on, it carries `aria-pressed`. A button that opened a popover or a menu carries `aria-expanded` until it closes. In both cases the button simply stays down, and the same affordance that invited the press now reports its state. A disabled button is flat and grey, because there is nothing to press. Busy is presentation only: the application sets `aria-busy` when a request starts and clears it when the request ends, and the button stays down meanwhile. It does not stop activation, so a control that must not be activated is `disabled`.

The default button is the primary action and takes the accent colour. `secondary` puts it on the neutral surface with a hairline border instead. `quiet` has no face, border, or shadow at rest and inherits the colour around it, for actions that should not compete with the content; it shows a wash of that colour when hovered, pressed, or latched.

A link that acts as a button, carries the `button` attribute. It takes every treatment a button does, including the variants. It cannot be disabled or busy, since a link has no such states.

In [ ]:
_button_css = r"""
:is(button, a[button]) {
  --face: var(--v); --_shift: 0px;
  --pad-block: var(--space-2); --pad-inline: var(--space);
  display: inline-flex; align-items: center; justify-content: center; gap: var(--space-2);
  padding: calc(var(--pad-block) + var(--_shift)) calc(var(--pad-inline) - var(--_shift))
           calc(var(--pad-block) - var(--_shift)) calc(var(--pad-inline) + var(--_shift));
  font-weight: var(--weight1); line-height: var(--leading-1);
  background: var(--face); color: var(--n-3); text-decoration: none;
  border: 1px solid transparent; border-radius: var(--radius);
  box-shadow: var(--bevel), var(--raised);
  cursor: pointer;
  transition: box-shadow calc(var(--motion) / 2) var(--ease), padding calc(var(--motion) / 2) var(--ease), background var(--motion) var(--ease);
}
:is(button, a[button]):hover { --face: var(--v1); }
:is(button, a[button]):is(:active, [aria-busy=true], [aria-pressed=true], [aria-expanded=true]) {
  --face: var(--v2); --_shift: calc(1px * var(--relief));
  box-shadow: var(--bevel-in), var(--sunken);
}
:is(button, a[button]):active { transition-duration: calc(var(--motion) / 4); }
:is(button, a[button]):disabled { --face: var(--n-1); --_shift: 0px; color: var(--n); box-shadow: var(--bevel); cursor: not-allowed; }
:is(button, a[button])[aria-pressed=true]:disabled { --face: var(--v-2); color: var(--v); }
button[aria-busy=true] { cursor: progress; }
button[aria-busy=true]::before {
  content: ''; inline-size: 1em; block-size: 1em;
  border: 2px solid currentColor; border-inline-end-color: transparent; border-radius: var(--radius-full);
  animation: aui-spin calc(var(--motion) * 5) linear infinite;
}
"""

Button = hypelt('Button', sel='button, a[button]', layer='components', doc='Raised at rest, down while active, busy, or expanded, latched by `aria-pressed`; `secondary` for the lesser action; a link carrying `button` is styled the same way.', css=_button_css)

In [ ]:
Button.variant('secondary', r"""
--face: var(--n-2); color: var(--n2); border-color: var(--n-1);
&:hover { --face: var(--v-3); }
&:is(:active, [aria-busy=true], [aria-pressed=true], [aria-expanded=true]) { --face: var(--n-2); }
&:is([aria-pressed=true], [aria-expanded=true]) { --face: var(--v-3); color: var(--v2); }
&:disabled { --face: var(--n-2); color: var(--n); }
&[aria-pressed=true]:disabled { --face: var(--v-2); color: var(--v); }
""", layer='components', doc='The lesser action: the neutral surface with a hairline border, and the accent wash when latched.')

Button.secondary [components] :is(button)[secondary]: The lesser action: the neutral surface with a hairline border, and the accent wash when latched.

In [ ]:
Button.variant('quiet', r'''
--face: transparent; color: inherit; border-color: transparent; box-shadow: none;
&:hover { --face: oklch(from currentColor l c h / .08); }
&:is(:active, [aria-busy=true], [aria-pressed=true], [aria-expanded=true]) { --face: oklch(from currentColor l c h / .14); --_shift: 0px; box-shadow: none; }
&:disabled { --face: transparent; color: var(--n); box-shadow: none; }
''', layer='components', doc='No face, border, or shadow at rest, for an action that should not compete: dismiss, close, and toolbar buttons.')

Button.quiet [components] :is(button)[quiet]: No face, border, or shadow at rest, for an action that should not compete: dismiss, close, and toolbar buttons.

Every face and state in a wrapping cluster: rest, secondary, a danger and a success context, disabled, busy, and latched:


In [ ]:
@example(Button)
def _button_example():
    return Cluster(Button('Save'), Button('Cancel', secondary=True), Button('Dismiss', quiet=True), A('Sign up', href='#', button=True), Button('Delete', danger=True), Button('Publish', success=True),
        Button('Locked', disabled=True), Button('Saving', aria_busy='true'), Button('Pinned', aria_pressed='true'), Button('Pinned', secondary=True, aria_pressed='true'), Button('Pinned', quiet=True, aria_pressed='true'),
        gap='var(--space-2)')


In [ ]:
Button.demo()

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 

Application styling can change the resting padding without rewriting the pressed state. Here `compact` changes the dimensions and `flat` removes relief. Declared as variants here, they belong to this example, not to the built-in button, and they appear in `doc(Button)` for as long as they are registered. To soften the whole application instead, set `:root { --relief: .5; }`.

In [ ]:
Button.variant('compact', '--pad-block: var(--space-3); --pad-inline: var(--space-2);', doc='Tighter padding.')
Button.variant('flat', '--relief: 0;', doc='No rims, shadows, or displacement.')

Button.flat [app] :is(button)[flat]: No rims, shadows, or displacement.

The flat buttons keep their colour states without rims, shadows, or displacement; the two pin buttons show unselected and selected states:

In [ ]:
demo(Cluster(
    Button('Compact', compact=True), Button('Flat', flat=True),
    Button('Pin', secondary=True, flat=True, aria_pressed='false'),
    Button('Pin', secondary=True, flat=True, aria_pressed='true'),
    gap='var(--space-2)'))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 

A button that holds nothing but an icon should be square, so when a button's only child is an `svg` we make its inline padding match its block padding. Give such a button an `aria-label`, because the icon gives it no name:

In [ ]:
Button.rule('icon-only', '&:has(> svg:only-child) { --pad-inline: var(--pad-block); }', layer='components', doc='A button whose only child is an svg is square; give it an aria-label.')

Button.icon-only [components] button: A button whose only child is an svg is square; give it an aria-label.

In [ ]:
cross = Svg(Path(d='M18 6 6 18'), Path(d='m6 6 12 12'), viewBox='0 0 24 24', fill='none', stroke='currentColor', stroke_width=2, stroke_linecap='round', stroke_linejoin='round', icon=True)
close = [Button(cross, aria_label='Close'), Button(cross, aria_label='Close', secondary=True), Button(cross, aria_label='Close', quiet=True)]
demo(Cluster(*close, Button(cross, 'Close', secondary=True), Button('Save'), gap='var(--space-2)'))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 

## Button group

Related actions, or a choice between a few views, sit better as one control than as a scatter of buttons. A button group is a row of buttons that share their edges: only the outer corners are rounded, and neighbours meet at a single hairline. The group is only layout. Which button is pressed is state, and the server should set this with `aria-pressed`.

In [ ]:
_button_group_css = r"""
button-group { display: inline-flex; }
button-group > button { border-radius: 0; }
button-group > button:first-child { border-start-start-radius: var(--radius); border-end-start-radius: var(--radius); }
button-group > button:last-child { border-start-end-radius: var(--radius); border-end-end-radius: var(--radius); }
button-group > button + button { margin-inline-start: -1px; }
"""

ButtonGroup = hypelt('ButtonGroup', layer='components', doc='A row of buttons sharing their edges; a latched secondary button takes the accent wash.', css=_button_group_css)

Paging actions, and a toggle bar with one button latched and one disabled. The bar is a named group, since its buttons are toggles rather than a form choice; an exclusive choice is what native radios are for.

In [ ]:
@example(ButtonGroup)
def _button_group_example():
    return Cluster(ButtonGroup(Button('Previous', secondary=True, type='button'), Button('Next', secondary=True, type='button')),
        ButtonGroup(Button('Bold', secondary=True, aria_pressed='true'), Button('Italic', secondary=True, aria_pressed='false'),
            Button('Underline', secondary=True, aria_pressed='false', disabled=True), role='group', aria_label='Text style'))

Note, a toggle's state is server state, so here we use a demo-scoped rule to stand in for the server and flip `aria-pressed` on click:

In [ ]:
reg.rule('demo-group-toggle', sel='button-group', on={('& > button[aria-pressed]', 'click'): r"el.setAttribute('aria-pressed', String(el.getAttribute('aria-pressed') !== 'true'))"},
    doc='Demo only: flips aria-pressed in place of the server.')

demo-group-toggle [app] button-group: Demo only: flips aria-pressed in place of the server.

In [ ]:
ButtonGroup.demo()

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 

## Text controls

Where a button is raised, a control is sunken. It is a well cut into the page, the same lighting as a button inverted, so the eye reads it as somewhere to put something rather than something to press. A disabled well goes flat, as a disabled button does. These rules apply to the tags themselves, so a form written by hand looks the same as one built with `Field`. The well is for the text-like input types, textareas, and selects; a button made from `input`, and a file picker, keep their native look.

Invalidity is a danger context. The browser sets `:user-invalid` once the user has interacted with a control, the server sets `aria-invalid`, and either one switches the control to the danger hue, so its border and error text take that colour with no rule of their own:

In [ ]:
reg.context(':user-invalid, [aria-invalid=true]', 'var(--hue-danger)')

_text_sel = 'input:is(:not([type]), [type=text], [type=email], [type=password], [type=search], [type=url], [type=tel], [type=number], [type=date], [type=time], [type=datetime-local], [type=month], [type=week])'
reg.rule('text-controls', rf"""
:is({_text_sel}, textarea, select) {{
  inline-size: 100%; padding: var(--space-2) var(--space-1);
  --face: var(--n-3); background: var(--face); color: var(--n2);
  border: 1px solid var(--n-1); border-radius: var(--radius); box-shadow: var(--bevel-in), var(--sunken);
  transition: border-color var(--motion) var(--ease), box-shadow var(--motion) var(--ease);
}}
:is(input, textarea, select):is(:user-invalid, [aria-invalid=true]) {{ border-color: var(--v); }}
:is({_text_sel}, textarea, select):disabled {{ --face: var(--n-2); color: var(--n); box-shadow: var(--bevel-in); cursor: not-allowed; }}
::placeholder {{ color: var(--n1); }}
""", layer='components', doc='The well shared by text inputs, textareas, and selects, taking the danger hue when invalid.')

Input = hypelt('Input', sel=_text_sel, layer='components', doc='A text-like input, styled as a well by the text-controls rule.')
Textarea = hypelt('Textarea', layer='components', css='textarea { min-block-size: 5rem; resize: vertical; field-sizing: content; }',
    doc='A textarea, styled as a well by the text-controls rule; it grows with its content.')
Select = hypelt('Select', layer='components', doc='A select, styled as a well by the text-controls rule.')

Each text control on its own, at rest, and disabled:

In [ ]:
@example(Input)
def _input_example(): return Stack(Input(placeholder='Search', type='search', aria_label='Search'), Input(value='Locked', disabled=True, aria_label='Locked'), style='inline-size:16rem')

@example(Textarea)
def _textarea_example(): return Textarea('A note that grows with its content.', aria_label='Note', style='inline-size:16rem')

@example(Select)
def _select_example(): return Select(Option('Free'), Option('Team'), aria_label='Plan', style='inline-size:16rem')

In [ ]:
Input.demo()

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 

## Field

A control rarely stands alone. It needs a label, often a line of help, and sometimes an error, and each of these has to be wired to the control so that assistive technology reads them together. `Field` does that wiring. It wraps the control in its label, which associates the two without an id, and puts a checkbox or radio before its label text rather than after. A `hint` and an `error` sit beneath, linked from the control's `aria-describedby`, and an error also marks the control `aria-invalid`. What counts as invalid is the application's decision; `Field` only shows the message it is given.


In [ ]:
_field_css = r"""
[field], [field] > label { display: grid; gap: var(--space-3); font-size: var(--text-1); }
[field] > label { font-weight: var(--weight1); }
[field] > small { font-weight: var(--weight); color: var(--n1); }
[field] > small[danger] { color: var(--v2); }
[field] > label:has(> :is([type=checkbox], [type=radio])) { grid-template-columns: auto 1fr; align-items: center; gap: var(--space-2); font-weight: var(--weight); }
[field]:has(:disabled) > label { color: var(--n); }
"""

@hypelt(sel='[field]', layer='components', css=_field_css)
def Field(
    label,      # Label text
    control,    # The control
    hint=None,  # Help text shown under the control
    error=None, # Error message; marks the control invalid
    **attrs,    # Attributes for the field
):
    "A labelled control with optional help and error text"
    boxed = control.attrs.get('type') in ('checkbox', 'radio')
    descriptions = []
    if hint: descriptions.append(Small(hint, id=True))
    if error:
        descriptions.append(Small(error, id=True, danger=True))
        control(aria_invalid='true')
    if descriptions:
        ids = [control.attrs.get('aria-describedby', ''), *(o.id for o in descriptions)]
        control(aria_describedby=' '.join(filter(None, ids)))
    return Div(Label(*([control, label] if boxed else [label, control])), *descriptions, field=True, **attrs)


Every kind of control in one form. The username keeps its hint alongside a server-supplied error. Type an invalid address into the email field and leave it to see the browser's validation treatment:

In [ ]:
@example(Field)
def _field_example():
    return Form(
        Field('Name', Input(name='name', placeholder='Ada Lovelace')),
        Field('Email', Input(name='email', type='email', required=True), hint='We never share it.'),
        Field('Notes', Textarea(name='notes', placeholder='Anything else?')),
        Field('Plan', Select(Option('Free'), Option('Team'), name='plan')),
        Field('Send me updates', Input(type='checkbox', name='updates', checked=True)),
        Field('Region', Input(name='region', value='Locked', disabled=True)),
        Field('Username', Input(name='username', value='taken'), hint='Shown on your profile.', error='That name is in use.'),
        Cluster(Button('Save'), Button('Cancel', secondary=True, type='button'), gap='var(--space-2)'),
        stack=True)


In [ ]:
Field.demo()

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 

Attributes passed to `Field` go on the field itself, so a variant declared on `Field` applies where its constructor puts the attribute. The control keeps its own attributes, and an existing description stays linked alongside the hint and error:

In [ ]:
notes = Textarea(name='notes', aria_describedby='notes-policy')
field = Field('Notes', notes, hint='Keep it brief.', error='Please remove the account number.', compact=True)
assert notes.attrs['aria-invalid'] == 'true'
assert notes.attrs['aria-describedby'].split() == ['notes-policy', *(o.id for o in field.children[1:])]
test_eq(field.attrs['compact'], True)
notes.attrs


{'name': 'notes',
 'aria-describedby': 'notes-policy _-C2HjYTtThGyL0xEn9xuTQ _8HVK8-FqSMqgEHQlGp6MXg',
 'aria-invalid': 'true'}

## Switch

Typically, a switch turns something on or off, doing so straight away, where a checkbox records a choice for a form to submit. The two are the same control to the browser: a switch is a checkbox with `role="switch"`, and the browser supplies the state, the keyboard handling, and the announcement. `Switch` only pairs the type with the role. Its look is built from the two treatments we already have: the track is a well, and the thumb is raised. Turning it on colours the track and slides the thumb across. In a `Field` it sits before its label, as a checkbox does:

In [ ]:
_switch_css = r"""
input[role=switch] { appearance: none; --face: var(--n-3); inline-size: 2.75em; block-size: 1.5em; margin: 0; padding: 2px;
  background: var(--face); border: 1px solid var(--n-1); border-radius: var(--radius-full); box-shadow: var(--bevel-in), var(--sunken);
  display: inline-grid; align-items: center; cursor: pointer;
  transition: background var(--motion) var(--ease), box-shadow var(--motion) var(--ease); }
input[role=switch]::before { content: ''; --face: var(--n-2); inline-size: calc(1.5em - 6px); block-size: calc(1.5em - 6px); border-radius: var(--radius-full);
  background: var(--face); box-shadow: var(--bevel), var(--raised);
  transition: translate var(--motion) var(--ease), background var(--motion) var(--ease); }
input[role=switch]:checked { --face: var(--v); }
input[role=switch]:checked::before { translate: 1.25em 0; --face: var(--n-3); }
input[role=switch]:disabled { --face: var(--n-2); box-shadow: var(--bevel-in); cursor: not-allowed; }
input[role=switch]:disabled::before { --face: var(--n-1); box-shadow: var(--bevel); }
"""

@hypelt(sel='input[role=switch]', layer='components', css=_switch_css)
def Switch(**attrs):
    "A checkbox with the switch role"
    return Input(type='checkbox', role='switch', **attrs)

Off, on, and disabled, then one in a field:

In [ ]:
@example(Switch)
def _switch_example():
    return Cluster(Switch(aria_label='Off'), Switch(checked=True, aria_label='On'), Switch(disabled=True, aria_label='Disabled'), Field('Email me about mentions', Switch(name='mentions', checked=True)))

In [ ]:
Switch.demo()

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 

## Checkbox and radio

The browser's own checkbox and radio follow the hue, but they have none of our depth, so we set `appearance: none` and restyle them ourselves. Each is a small well. At this size we drop the rims and keep only the sunken shade. `Checkbox` and `Radio` only set the type:

In [ ]:
_choice_css = r"""
input[type=checkbox]:not([role=switch]), input[type=radio] { appearance: none; --face: var(--n-3); inline-size: 1.15em; block-size: 1.15em; margin: 0;
  background: var(--face); border: 1px solid var(--n-1); border-radius: var(--radius-1); box-shadow: var(--sunken);
  display: inline-grid; place-content: center; cursor: pointer;
  transition: background var(--motion) var(--ease), border-color var(--motion) var(--ease); }
input[type=checkbox]:not([role=switch])::before, input[type=radio]::before { content: ''; inline-size: .6em; block-size: .6em; background: var(--n-3); scale: 0; transition: scale var(--motion) var(--ease); }
:is(input[type=checkbox]:not([role=switch]), input[type=radio]):disabled { --face: var(--n-2); border-color: var(--n-1); cursor: not-allowed; }
"""
reg.rule('choice-controls', _choice_css, layer='components', doc='The small well and the mark shared by checkboxes and radios.')

_checkbox_css = r"""
input[type=checkbox]:not([role=switch])::before { clip-path: polygon(14% 44%, 0 65%, 50% 100%, 100% 16%, 80% 0%, 43% 62%); }
input[type=checkbox]:not([role=switch]):is(:checked, :indeterminate) { --face: var(--v); border-color: var(--v); }
input[type=checkbox]:not([role=switch]):is(:checked, :indeterminate)::before { scale: 1; }
input[type=checkbox]:not([role=switch]):indeterminate::before { clip-path: none; block-size: .15em; }
input[type=checkbox]:not([role=switch]):is(:checked, :indeterminate):disabled { --face: var(--n-1); border-color: var(--n-1); }
"""

@hypelt(sel='input[type=checkbox]:not([role=switch])', layer='components', css=_checkbox_css)
def Checkbox(**attrs):
    "A checkbox; the choice-controls rule draws its well"
    return Input(type='checkbox', **attrs)


In [ ]:
_radio_css = r"""
input[type=radio] { border-radius: var(--radius-full); }
input[type=radio]::before { border-radius: var(--radius-full); background: var(--v); }
input[type=radio]:checked { border-color: var(--v); }
input[type=radio]:checked::before { scale: 1; }
input[type=radio]:checked:disabled::before { background: var(--n); }
"""

@hypelt(sel='input[type=radio]', layer='components', css=_radio_css)
def Radio(**attrs):
    "A radio button; the choice-controls rule draws its well"
    return Input(type='radio', **attrs)

Unchecked, checked, disabled, and checked and disabled, then one in a field:

In [ ]:
@example(Checkbox)
def _checkbox_example():
    return Cluster(Checkbox(aria_label='Unchecked'), Checkbox(checked=True, aria_label='Checked'),
        Checkbox(disabled=True, aria_label='Disabled'), Checkbox(checked=True, disabled=True, aria_label='Checked and disabled'),
        Field('Email me about mentions', Checkbox(name='mentions', checked=True)))


In [ ]:
Checkbox.demo()

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 

Radios in a group, unchecked, checked, and checked and disabled, then one in a field:

In [ ]:
@example(Radio)
def _radio_example():
    return Cluster(Radio(name='r', aria_label='Unchecked'), Radio(name='r', checked=True, aria_label='Checked'), Radio(name='rd', checked=True, disabled=True, aria_label='Checked and disabled'),
        Field('Weekly', Radio(name='digest', checked=True)))

In [ ]:
Radio.demo()

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 

## Range

A slider is the browser's `input[type=range]`, restyled as a well with a raised thumb. The one thing we add is a fill along the track up to the thumb. CSS cannot know where the thumb is, so a small behavior reads the slider's value whenever it changes and writes the fill, computed from the control's own bounds, into `--p`. A value set from a script does not fire that event, so such a change dispatches `input` to update the fill.

In [ ]:
_fill = r"const lo = el.min === '' ? 0 : +el.min, hi = el.max === '' ? 100 : +el.max; el.style.setProperty('--p', (hi > lo ? (el.value - lo) / (hi - lo) * 100 : 0) + '%')"
_range_on = {
    ('&', ':load'): _fill,
    ('&', 'input'): _fill,
    ('form', 'reset'): r"""
await new Promise(r => setTimeout(r, 0));
for (const range of [...el.elements].filter(i => i.matches('input[type=range]'))) { const el = range; """ + _fill + " }"}

_range_css = r"""
input[type=range] { appearance: none; inline-size: 100%; block-size: 1.5em; margin: 0; background: none; cursor: pointer;
  --track: .6em; --thumb: 1.25em; --p: 0%; }
input[type=range]::-webkit-slider-runnable-track, input[type=range]::-webkit-slider-thumb {
  --hi: oklch(from var(--face) calc(l + .16) c h / var(--relief)); --lo: oklch(from var(--face) calc(l - .16) c h / var(--relief));
  --bevel: inset 1px 1px 0 var(--hi), inset -1px -1px 0 var(--lo); --bevel-in: inset 1px 1px 0 var(--lo), inset -1px -1px 0 var(--hi); }
input[type=range]::-webkit-slider-runnable-track { --face: var(--n-3); block-size: var(--track); border-radius: var(--radius-full); border: 1px solid var(--n-1);
  background: linear-gradient(to right, var(--v) var(--p), var(--face) 0); box-shadow: var(--bevel-in), var(--sunken); }
input[type=range]::-webkit-slider-thumb { appearance: none; --face: var(--v); inline-size: var(--thumb); block-size: var(--thumb);
  margin-block-start: calc((var(--track) - var(--thumb)) / 2 - 1px); border-radius: var(--radius-full);
  background: var(--face); box-shadow: var(--bevel), var(--raised); transition: box-shadow calc(var(--motion) / 2) var(--ease); }
input[type=range]:active::-webkit-slider-thumb { box-shadow: var(--bevel-in), var(--sunken); }
input[type=range]:disabled { cursor: not-allowed; }
input[type=range]:disabled::-webkit-slider-runnable-track { background: linear-gradient(to right, var(--n) var(--p), var(--n-2) 0); }
input[type=range]:disabled::-webkit-slider-thumb { --face: var(--n-1); box-shadow: var(--bevel); }
"""

@hypelt(sel='input[type=range]', layer='components', on=_range_on, css=_range_css)
def Range(**attrs):
    "A slider"
    return Input(type='range', **attrs)

A slider at 35, a disabled one at 65, one at 8 of 10, and one at -5 in a range that ends at zero, where a missing bound and a bound of zero must not be confused. Volume has no explicit value, so the browser starts it at the midpoint.

In [ ]:
@example(Range)
def _range_example():
    return Stack(Range(value=35, aria_label='Brightness'), Range(value=65, disabled=True, aria_label='Locked'), Range(min=0, max=10, value=8, aria_label='Rating'),
        Range(min=-10, max=0, value=-5, aria_label='Offset'), Field('Volume', Range(name='volume')),
        style='inline-size:16rem')

In [ ]:
Range.demo()

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 

## Fieldset

A fieldset groups fields under a legend. It draws a hairline panel and owns its padding; spacing between its fields is the layout's job, as everywhere else:

In [ ]:
_fieldset_css = r"""
fieldset { margin: 0; min-inline-size: 0; border: 1px solid var(--n-1); border-radius: var(--radius); padding: var(--space); }
legend { font-weight: var(--weight1); padding-inline: var(--space-2); }
"""

Fieldset = hypelt('Fieldset', layer='components', doc='A bordered group of fields under a legend.', css=_fieldset_css)

Two checkbox fields under a legend:

In [ ]:
@example(Fieldset)
def _fieldset_example():
    return Fieldset(Legend('Notifications'),
        Field('Email me about mentions', Input(type='checkbox', checked=True)),
        Field('Email me a weekly digest', Input(type='checkbox')),
        stack=True, style='--gap: var(--space-2)')

In [ ]:
Fieldset.demo()

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer reset {
/* reset */
*, *::before, *::after { box-sizing: border-box; }
:where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, figure, figcaption, pre, hr, table) { margin: 0; }
img, picture, video, canvas, svg { max-inline-size: 100%; block-size: auto; }
button, input, textarea, select { font: inherit; color: inherit; }
}

@layer tokens {
/* hue */
:root { --hue: 260; color-scheme: light dark; }
/* contexts */
:root { --hue-danger: 25; --hue-warning: 75; --hue-success: 145; --hue-info: 230; }
/* palette */
[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }
mark { --hue: var(--hue-warning); }
:user-invalid, [aria-invalid=true] { --hue: var(--hue-danger); }
:root, [danger], [warning], [success], [info], mark, :user-invalid, [aria-invalid=true] {
--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.00329*cos((2*var(--hue) - 268.97)*1deg) + 0.00139*cos((3*var(--hue) - 327.76)*1deg) + 0.00123*cos((4*var(--hue) - 93.61)*1deg) + 0.00125*cos((5*var(--hue) - 211.17)*1deg) + 0.00034*cos((6*var(--hue) - 339.8)*1deg) + 0.00023*cos((7*var(--hue) - 56.25)*1deg) + 0.00038*cos((8*var(--hue) - 125.04)*1deg) + 0.00015*cos((9*var(--hue) - 283.35)*1deg) + 7e-05*cos((10*var(--hue) - 349.22)*1deg) + 0.00021*cos((11*var(--hue) - 12.11)*1deg) + 0.00015*cos((12*var(--hue) - 161.94)*1deg), 0.02) var(--hue)), oklch(0.1026 clamp(0, 0.01959 + 0.00041*cos((1*var(--hue) - 19.99)*1deg), 0.02) var(--hue))); /* base: the lightness extreme (the default page) */
--n-2: light-dark(oklch(0.9506 0.02 var(--hue)), oklch(0.1434 0.02 var(--hue))); /* raised surface: cards, inputs */
--n-1: light-dark(oklch(0.7903 0.02 var(--hue)), oklch(0.3385 0.02 var(--hue))); /* hairline borders, separators */
--n: light-dark(oklch(0.6503 0.02 var(--hue)), oklch(0.4611 0.02 var(--hue))); /* disabled text and controls */
--n1: light-dark(oklch(0.5247 0.02 var(--hue)), oklch(0.5812 0.02 var(--hue))); /* muted and placeholder text */
--n2: light-dark(oklch(0.4217 0.02 var(--hue)), oklch(0.693 0.02 var(--hue))); /* body text; body text on surface */
--v-3: light-dark(oklch(0.913 0.03581 var(--hue)), oklch(0.1825 clamp(0, 0.05701 + 0.02563*cos((1*var(--hue) - 325.45)*1deg) + 0.01252*cos((2*var(--hue) - 263.8)*1deg) + 0.00819*cos((3*var(--hue) - 96.1)*1deg) + 0.00256*cos((4*var(--hue) - 85.33)*1deg) + 0.0017*cos((5*var(--hue) - 319.1)*1deg) + 0.0034*cos((6*var(--hue) - 157.26)*1deg) + 0.00073*cos((7*var(--hue) - 138.05)*1deg) + 0.00112*cos((8*var(--hue) - 2.43)*1deg) + 0.00177*cos((9*var(--hue) - 237.19)*1deg) + 0.00037*cos((10*var(--hue) - 228.33)*1deg) + 0.00064*cos((11*var(--hue) - 69.15)*1deg), 0.09221) var(--hue))); /* subtle background wash; distinct from raised surface */
--v-2: light-dark(oklch(0.7453 clamp(0, 0.12952 + 0.00279*cos((1*var(--hue) - 18.52)*1deg), 0.13231) var(--hue)), oklch(0.3792 clamp(0, 0.11554 + 0.05156*cos((1*var(--hue) - 327.56)*1deg) + 0.02507*cos((2*var(--hue) - 264.93)*1deg) + 0.01578*cos((3*var(--hue) - 92.55)*1deg) + 0.00467*cos((4*var(--hue) - 86.48)*1deg) + 0.00337*cos((5*var(--hue) - 312.62)*1deg) + 0.00706*cos((6*var(--hue) - 156.72)*1deg) + 0.00165*cos((7*var(--hue) - 144.83)*1deg) + 0.00212*cos((8*var(--hue) - 5.61)*1deg) + 0.00327*cos((9*var(--hue) - 233.15)*1deg), 0.18274) var(--hue))); /* wash hover, selected */
--v-1: light-dark(oklch(0.6068 clamp(0, 0.15177 

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()